In [1]:
# Measure the time it takes every cell to run
%pip install ipython-autotime
%load_ext autotime

time: 265 µs (started: 2026-03-23 20:43:02 +00:00)


In [2]:
# Set up idc-index, used to make clinical table parameters human-readable
%pip install idc-index
from idc_index import IDCClient
c = IDCClient()
c.fetch_index('clinical_index')

time: 20 s (started: 2026-03-23 20:43:04 +00:00)


In [3]:
# Set up BigQuery, used to construct patient cohort of interest
import os
my_ProjectID = "nlst-radiomics"
os.environ["GCP_PROJECT_ID"] = my_ProjectID

from google.colab import auth, files
auth.authenticate_user()

from google.cloud import bigquery
bq_client = bigquery.Client(my_ProjectID)

time: 6.77 s (started: 2026-03-23 20:43:47 +00:00)


In [4]:
# Set up pyradiomics and pydicom_seg
%pip install git+https://github.com/Radiomics/pyradiomics.git
%pip install SimpleITK

import SimpleITK as sitk
from radiomics import featureextractor

  Cloning https://github.com/Radiomics/pyradiomics.git to /tmp/pip-req-build-97c562qd
  Running command git clone --filter=blob:none --quiet https://github.com/Radiomics/pyradiomics.git /tmp/pip-req-build-97c562qd
  Resolved https://github.com/Radiomics/pyradiomics.git to commit 8ed579383b44806651c463d5e691f3b2b57522ab
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
time: 25.5 s (started: 2026-03-23 20:43:57 +00:00)


In [5]:
# First, what are the possible segmentation types?
# Note: I checked that 'AIMI lung and nodule AI segmentation' contains 'Lung' and 'Nodule' segment labels

seg_query = """
SELECT SeriesDescription
FROM `bigquery-public-data.idc_v23.dicom_all`
WHERE
  collection_id = 'nlst'
  AND Modality = 'SEG'
"""

seg_result = bq_client.query(seg_query)
seg_metadata_df = seg_result.result().to_dataframe()

print(f"\n--- Unique Series Descriptions ---")
display(seg_metadata_df['SeriesDescription'].value_counts().head(20))

print(f"\n--- Unique Series Descriptions Containing 'lung and nodule' ---")
lung_nodule_segmentation_types = seg_metadata_df[seg_metadata_df['SeriesDescription'].str.contains('lung and nodule', case=False)]['SeriesDescription'].unique().tolist()
display(lung_nodule_segmentation_types)


--- Unique Series Descriptions ---


,count
SeriesDescription,
TotalSegmentator(v1.5.6) Segmentation of Series 2,47987
TotalSegmentator(v1.5.6) Segmentation of Series 3,38198
TotalSegmentator(v1.5.6) Segmentation of Series 4,13362
TotalSegmentator(v1.5.6) Segmentation of Series 5,6180
TotalSegmentator(v1.5.6) Segmentation of Series 6,5377
TotalSegmentator(v1.5.6) Segmentation of Series 1,4449
TotalSegmentator(v1.5.6) Segmentation of Series 0,1132
AIMI lung and nodule AI segmentation,1042
3d_fullres-tta_nnU-Net_Segmentation,1039



--- Unique Series Descriptions Containing 'lung and nodule' ---


['AIMI lung and nodule radiologist 8 corrected segmentation',
 'AIMI lung and nodule radiologist 5 corrected segmentation',
 'AIMI lung and nodule radiologist 4 corrected segmentation',
 'AIMI lung and nodule AI segmentation']

time: 2.86 s (started: 2026-03-23 20:44:28 +00:00)


In [6]:
# How many CT/SEG pairs with lung and nodule segmentations are there?
# Note: If seg.SeriesDescription is not radiologist corrected, there are 1042 SEG files with lung and nodule segmentations

ct_seg_join_query = f"""
SELECT
  DISTINCT seg.SeriesInstanceUID AS SEG_SeriesInstanceUID,
  ct.PatientID,
  ct.SeriesInstanceUID AS CT_SeriesInstanceUID,
  seg.SeriesDescription AS SEG_SeriesDescription,
  seg.StudyDate,
  CASE
    WHEN FORMAT_DATE('%Y', seg.StudyDate) LIKE '1999%' THEN 'T0'
    WHEN FORMAT_DATE('%Y', seg.StudyDate) LIKE '2000%' THEN 'T1'
    WHEN FORMAT_DATE('%Y', seg.StudyDate) LIKE '2001%' THEN 'T2'
    ELSE 'Other'
  END AS StudyYear
FROM
  `bigquery-public-data.idc_v23.dicom_all` AS ct
JOIN
  `bigquery-public-data.idc_v23.dicom_all` AS seg
ON
  ct.collection_id = seg.collection_id
  AND ct.PatientID = seg.PatientID
WHERE
  ct.collection_id = 'nlst'
  AND ct.Modality = 'CT'
  AND seg.Modality = 'SEG'
  AND seg.SeriesDescription IN ({', '.join([f"'{s}'" for s in lung_nodule_segmentation_types])})
  AND EXISTS (
    SELECT 1
    FROM UNNEST(seg.ReferencedSeriesSequence) AS ref_seq
    WHERE ct.SeriesInstanceUID = ref_seq.SeriesInstanceUID
  )
"""

ct_seg_join_result = bq_client.query(ct_seg_join_query)
ct_seg_join_df = ct_seg_join_result.result().to_dataframe()

print(f"Number of distinct patients with a CT/SEG pair: {len(ct_seg_join_df['PatientID'].unique())}")
print(f"Number of unique CT series: {len(ct_seg_join_df['CT_SeriesInstanceUID'].unique())}")
print(f"Number of CT/SEG pairs: {ct_seg_join_df.shape[0]}")
print(f"Number of distinct study dates: {len(ct_seg_join_df['StudyDate'].unique())}")

Number of distinct patients with a CT/SEG pair: 572
Number of unique CT series: 1042
Number of CT/SEG pairs: 1144
Number of distinct study dates: 3
time: 2.13 s (started: 2026-03-23 20:44:34 +00:00)


In [7]:
# Two SEG series can correspond to one CT series because of radiologist corrected segmentations. "Keep" the CT/SEG pair with the radiologist corrected segmentation and throw out the other.

priority_segmentation_type = 'AIMI lung and nodule AI segmentation'

# Create a priority column: 0 for non-AI, 1 for AI
ct_seg_join_df['priority'] = ct_seg_join_df['SEG_SeriesDescription'].apply(lambda x: 1 if x == priority_segmentation_type else 0)

# Sort by CT_SeriesInstanceUID and then by priority (0 comes before 1, so non-AI is prioritized)
ct_seg_join_df_filtered = ct_seg_join_df.sort_values(by=['CT_SeriesInstanceUID', 'priority'])

# Drop duplicates, keeping the first (which will be the non-AI one if present, otherwise the first AI one)
ct_seg_join_df_one_to_one = ct_seg_join_df_filtered.drop_duplicates(subset='CT_SeriesInstanceUID', keep='first')

print(f"\nNumber of CT/SEG pairs after filtering: {ct_seg_join_df_one_to_one.shape[0]}")
display(ct_seg_join_df_one_to_one.head())


Number of CT/SEG pairs after filtering: 1042


,SEG_SeriesInstanceUID,PatientID,CT_SeriesInstanceUID,SEG_SeriesDescription,StudyDate,StudyYear,priority
275,1.2.276.0.7230010.3.1.3.17436516.3112497.17206...,107002,1.2.840.113654.2.55.10057224705584082952969157...,AIMI lung and nodule AI segmentation,1999-01-02,T0,1
348,1.2.276.0.7230010.3.1.3.17436516.3112767.17206...,107030,1.2.840.113654.2.55.10087518978221069034420730...,AIMI lung and nodule AI segmentation,1999-01-02,T0,1
328,1.2.276.0.7230010.3.1.3.17436516.3182127.17206...,126955,1.2.840.113654.2.55.10212803674068251180698456...,AIMI lung and nodule AI segmentation,2000-01-02,T1,1
679,1.2.276.0.7230010.3.1.3.17436516.3103212.17206...,104999,1.2.840.113654.2.55.10222078521568314836955069...,AIMI lung and nodule AI segmentation,1999-01-02,T0,1
778,1.2.276.0.7230010.3.1.3.17436516.3180509.17206...,126446,1.2.840.113654.2.55.10223655654547815709262155...,AIMI lung and nodule radiologist 5 corrected s...,1999-01-02,T0,0


time: 29.7 ms (started: 2026-03-23 20:44:38 +00:00)


In [8]:
# Download CT/SEG pairs for 5 distinct patients using idc_index

# First, ensure the download directory is set on the Colab VM
download_dir_on_colab_vm = "/content/malignant_trial"
if not os.path.exists(download_dir_on_colab_vm):
    os.makedirs(download_dir_on_colab_vm)

# Your existing code to get series IDs
five_malignant_patientIDs = ct_seg_join_df_one_to_one['PatientID'].unique().tolist()[:5]
CT_SEG_seriesIDs = ct_seg_join_df_one_to_one[ct_seg_join_df_one_to_one['PatientID'].isin(five_malignant_patientIDs)][['CT_SeriesInstanceUID', 'SEG_SeriesInstanceUID']].values.flatten().tolist()

print(f"Downloading {len(CT_SEG_seriesIDs)} series for {len(five_malignant_patientIDs)} patients to Colab VM at {download_dir_on_colab_vm}...")
c.download_from_selection(seriesInstanceUID=CT_SEG_seriesIDs, downloadDir=download_dir_on_colab_vm)
print("Download to Colab VM complete.")

Download to Colab VM complete.
time: 10.5 s (started: 2026-03-23 20:44:41 +00:00)


In [9]:
# Compress the downloaded directory into a zip file
#zip_filename = 'malignant_trial_data.zip'
#!zip -r "{zip_filename}" "{download_dir_on_colab_vm}"

# Trigger download to your local machine
#print(f"\nTriggering download of '{zip_filename}' to your local machine...")
#files.download(zip_filename)
#print("Download initiated. Please check your browser's downloads.")

time: 431 µs (started: 2026-03-23 20:44:59 +00:00)


In [10]:
# Convert CT and SEG series to .nrrd files for each patient and timestep
base_path = '/content/malignant_trial/nlst'

for patient_id in os.listdir(base_path):
    patient_path = os.path.join(base_path, patient_id)
    if not os.path.isdir(patient_path):
        continue

    for timestep_dir in os.listdir(patient_path):
        timestep_path = os.path.join(patient_path, timestep_dir)
        if not os.path.isdir(timestep_path):
            continue

        ct_series_path = None
        seg_series_path = None

        for item in os.listdir(timestep_path):
            full_item_path = os.path.join(timestep_path, item)
            if os.path.isdir(full_item_path):
                if item.startswith('CT_'):
                    ct_series_path = full_item_path
                elif item.startswith('SEG_'):
                    seg_series_path = full_item_path

        if ct_series_path:
            try:
                reader = sitk.ImageSeriesReader()
                dicom_names = reader.GetGDCMSeriesFileNames(ct_series_path)
                if dicom_names:
                    reader.SetFileNames(dicom_names)
                    ct_image = reader.Execute()
                    output_ct_path = os.path.join(timestep_path, 'CT.nrrd')
                    sitk.WriteImage(ct_image, output_ct_path)
                    print(f"Converted CT for patient {patient_id}, timestep {timestep_dir} to {output_ct_path}")
                else:
                    print(f"No DICOM files found in CT series for patient {patient_id}, timestep {timestep_dir}")
            except Exception as e:
                print(f"Failed to convert CT for patient {patient_id}, timestep {timestep_dir}: {e}")
        else:
            print(f"CT series folder not found for patient {patient_id}, timestep {timestep_dir}")

        if seg_series_path:
            try:
                seg_files = [os.path.join(seg_series_path, f) for f in os.listdir(seg_series_path) if f.endswith('.dcm')]
                if seg_files:
                    # Assuming there's one primary DICOM SEG file in the folder for conversion
                    seg_dcm_path = seg_files[0]
                    reader = sitk.ImageFileReader()
                    reader.SetFileName(seg_dcm_path)
                    seg_image = reader.Execute()
                    output_seg_path = os.path.join(timestep_path, 'SEG.nrrd')
                    sitk.WriteImage(seg_image, output_seg_path, useCompression=True)
                    print(f"Converted SEG for patient {patient_id}, timestep {timestep_dir} to {output_seg_path}")
                else:
                    print(f"No DICOM SEG files found in SEG series for patient {patient_id}, timestep {timestep_dir}")
            except Exception as e:
                print(f"Failed to convert SEG for patient {patient_id}, timestep {timestep_dir}: {e}")
        else:
            print(f"SEG series folder not found for patient {patient_id}, timestep {timestep_dir}")

Converted CT for patient 126446, timestep 1.2.840.113654.2.55.257572809223995827394443746663383442894 to /content/malignant_trial/nlst/126446/1.2.840.113654.2.55.257572809223995827394443746663383442894/CT.nrrd
Converted SEG for patient 126446, timestep 1.2.840.113654.2.55.257572809223995827394443746663383442894 to /content/malignant_trial/nlst/126446/1.2.840.113654.2.55.257572809223995827394443746663383442894/SEG.nrrd
Converted CT for patient 107002, timestep 1.2.840.113654.2.55.35671975265195223904685112987162619590 to /content/malignant_trial/nlst/107002/1.2.840.113654.2.55.35671975265195223904685112987162619590/CT.nrrd
Converted SEG for patient 107002, timestep 1.2.840.113654.2.55.35671975265195223904685112987162619590 to /content/malignant_trial/nlst/107002/1.2.840.113654.2.55.35671975265195223904685112987162619590/SEG.nrrd
Converted CT for patient 107002, timestep 1.2.840.113654.2.55.136072716603498717401162853674652254104 to /content/malignant_trial/nlst/107002/1.2.840.113654.2.5

In [ ]:
# pyradiomics feature extraction
path = '/content/malignant_trial/nlst'
target = 'SEG' # name of segmentation mask (ITV.nrrd)........ change????
out_path = '/content/malignant_trial_features'

new_ps = [1.0]*3
print('Isotropic resampling to', new_ps)

settings = {}
settings['resegmentRange'] = [-1000,2000] # excludes air and bone (HU scale)
settings['resampledPixelSpacing'] = new_ps
settings['interpolator'] = 'sitkLanczosWindowedSinc' # for CT (lanczos is most reproducible)

print('Extracting radiomic features')
extractor = featureextractor.RadiomicsFeatureExtractor(**settings)

dataFrames = []
for patient in os.listdir(path):
    if not os.path.isdir(os.path.join(path, patient)):
        continue
    for timestep in os.listdir(os.path.join(path, patient)):
        mask = os.path.join(path, patient, timestep, target + '.nrrd')
        ct = os.path.join(path, patient, timestep, 'CT.nrrd')
        try:
            features = extractor.execute(ct, mask)
            df = pd.DataFrame.from_dict({i:str(features[i]) for i in features}, orient='index', columns=[patient+timestep])
            dataFrames.append(df)
            print('Extracted features from ', patient, timestep)
        except:
            print('Failed to extract features from ', patient, timestep)